In [ ]:
# Input: 427_metabolites_and_abundance.xlsx, sheet Merged; place the file alongside this notebook.
# Input columns: Metaphlan_ID, Match_method, Sample_ID, Full_name, TA, IAM, ILA, I3C, IE, IAA, IA, IPA, INDOLE, followed by numeric species abundances.
# Sample IDs must be unique; their prefixes identify cohorts. Abundances and measured metabolite levels must be non-negative.

from __future__ import annotations

import json
import math
import platform
import warnings
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
import scipy
import xgboost
from IPython.display import display
from scipy.stats import spearmanr
from sklearn.base import clone
from sklearn.cross_decomposition import PLSRegression
from sklearn.ensemble import (ExtraTreesRegressor, GradientBoostingRegressor,
                              HistGradientBoostingRegressor, RandomForestRegressor)
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import ElasticNet, Lasso, Ridge
from sklearn.metrics import r2_score
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=ConvergenceWarning)
sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)

INPUT_NAME = '427_metabolites_and_abundance.xlsx'
OUTPUT_FOLDER_NAME = 'Results'
SHEET_NAME = 'Merged'
TARGETS = ['TA', 'IAM', 'ILA', 'I3C', 'IE', 'IAA', 'IA', 'IPA', 'INDOLE']
METADATA_COLUMNS = ['Metaphlan_ID', 'Match_method', 'Sample_ID', 'Full_name']
N_SPLITS = 5
N_REPEATS = 10
CV_RANDOM_STATE = 20260806
MODEL_RANDOM_STATE = 20260806
LOG_PSEUDOCOUNT = 1.0

INPUT_PATH = Path(INPUT_NAME)
OUTPUT_DIR = Path(OUTPUT_FOLDER_NAME)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Input:', INPUT_PATH)
print('Output:', OUTPUT_DIR)
print(f'CV: {N_REPEATS} repeats x {N_SPLITS} folds; stratified by source label; random_state={CV_RANDOM_STATE}')


In [ ]:
data = pd.read_excel(INPUT_PATH, sheet_name=SHEET_NAME)
required = set(METADATA_COLUMNS + TARGETS)
missing_required = sorted(required - set(data.columns))
if missing_required:
    raise ValueError(f'Missing required columns: {missing_required}')
if data['Metaphlan_ID'].isna().any() or data['Metaphlan_ID'].duplicated().any():
    raise ValueError('Metaphlan_ID must be nonmissing and unique.')

feature_columns = [c for c in data.columns if c not in required]
if not feature_columns:
    raise ValueError('No species feature columns were identified.')

def cohort_from_id(sample_id: object) -> str:
    return str(sample_id).split('_', 1)[0]

sample_ids = data['Metaphlan_ID'].astype(str).to_numpy()
cohorts = np.array([cohort_from_id(x) for x in sample_ids], dtype=object)

X_raw_df = data[feature_columns].apply(pd.to_numeric, errors='coerce')
negative_x = int((X_raw_df < 0).sum().sum())
if negative_x:
    raise ValueError(f'Found {negative_x} negative species abundances; log2(x+1) transformation is not supported.')
X_log2 = np.log2(X_raw_df.fillna(0.0).to_numpy(dtype=float) + LOG_PSEUDOCOUNT)

target_audit_rows = []
for target in TARGETS:
    y_raw = pd.to_numeric(data[target], errors='coerce')
    valid = y_raw.notna() & np.isfinite(y_raw)
    if (y_raw[valid] < 0).any():
        raise ValueError(f'{target} contains negative values; log2(y+1) transformation is not supported.')
    counts = pd.Series(cohorts[valid.to_numpy()]).value_counts().sort_index()
    if counts.min() < N_SPLITS:
        raise ValueError(f'{target} has a source group with fewer than {N_SPLITS} valid samples: {counts.to_dict()}')
    target_audit_rows.append({
        'Target': target,
        'N': int(valid.sum()),
        'Missing_y': int((~valid).sum()),
        'Zero_y': int((y_raw[valid] == 0).sum()),
        'N_Cohorts': int(counts.size),
        'Cohort_Counts': '; '.join(f'{k}:{v}' for k, v in counts.items()),
    })

target_audit = pd.DataFrame(target_audit_rows)
cohort_counts = pd.Series(cohorts, name='Cohort').value_counts().sort_index().rename_axis('Cohort').reset_index(name='N')

print(f'Samples: {len(data)}; species features: {len(feature_columns)}; missing species abundances filled with zero: {int(X_raw_df.isna().sum().sum()):,}')
display(cohort_counts)
display(target_audit)


In [ ]:
def make_model_templates() -> dict[str, object]:
    return {
        'KNN': KNeighborsRegressor(n_neighbors=15, weights='distance', p=2),
        'SVR_RBF': SVR(C=10.0, epsilon=0.1, gamma='scale', kernel='rbf'),
        'Lasso': Lasso(alpha=0.01, max_iter=30000, selection='cyclic'),
        'ElasticNet': ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=30000, selection='cyclic'),
        'Ridge': Ridge(alpha=10.0),
        'PLS': PLSRegression(n_components=10, scale=False, max_iter=1000),
        'RandomForest': RandomForestRegressor(
            n_estimators=250,
            max_features=0.5,
            min_samples_leaf=2,
            random_state=MODEL_RANDOM_STATE,
            n_jobs=-1,
        ),
        'ExtraTrees': ExtraTreesRegressor(
            n_estimators=250, max_features=1.0, min_samples_leaf=2,
            random_state=MODEL_RANDOM_STATE, n_jobs=-1,
        ),
        'GradientBoosting': GradientBoostingRegressor(
            n_estimators=150, learning_rate=0.05, max_depth=2,
            min_samples_leaf=3, loss='huber', random_state=MODEL_RANDOM_STATE,
        ),
        'HistGradientBoosting': HistGradientBoostingRegressor(
            max_iter=150,
            learning_rate=0.05,
            max_leaf_nodes=15,
            min_samples_leaf=10,
            l2_regularization=1.0,
            random_state=MODEL_RANDOM_STATE,
        ),
        'XGBoost': XGBRegressor(
            n_estimators=300,
            learning_rate=0.03,
            max_depth=3,
            min_child_weight=5,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.1,
            reg_lambda=1.0,
            objective='reg:squarederror',
            eval_metric='rmse',
            tree_method='hist',
            random_state=MODEL_RANDOM_STATE,
            n_jobs=-1,
        ),
    }

def safe_spearman(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    if np.unique(y_true).size < 2 or np.unique(y_pred).size < 2:
        return math.nan
    return float(spearmanr(y_true, y_pred).statistic)

def seeded_clone(estimator: object, seed: int) -> object:
    model = clone(estimator)
    if 'random_state' in model.get_params(deep=False):
        model.set_params(random_state=seed)
    return model

model_templates = make_model_templates()
MODEL_NAMES = list(model_templates)
print(f'Number of candidate models: {len(MODEL_NAMES)}')
MODEL_NAMES


In [ ]:
fold_rows = []
repeat_rows = []
prediction_store = {}
fold_id_store = {}
valid_index_store = {}
y_store = {}
cohort_store = {}

for target_number, target in enumerate(TARGETS, start=1):
    y_raw = pd.to_numeric(data[target], errors='coerce').to_numpy(dtype=float)
    valid_mask = np.isfinite(y_raw)
    valid_global = np.flatnonzero(valid_mask)
    y_valid = np.log2(y_raw[valid_mask] + LOG_PSEUDOCOUNT)
    cohort_valid = cohorts[valid_mask]
    X_valid = X_log2[valid_mask]
    splitter = RepeatedStratifiedKFold(
        n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=CV_RANDOM_STATE
    )
    splits = list(splitter.split(X_valid, cohort_valid))
    valid_index_store[target] = valid_global
    y_store[target] = y_valid
    cohort_store[target] = cohort_valid

    for model_number, model_name in enumerate(MODEL_NAMES, start=1):
        repeat_buffers = {r: np.full(len(y_valid), np.nan) for r in range(1, N_REPEATS + 1)}
        repeat_fold_ids = {r: np.zeros(len(y_valid), dtype=int) for r in range(1, N_REPEATS + 1)}

        for split_number, (train_local, test_local) in enumerate(splits):
            repeat = split_number // N_SPLITS + 1
            fold = split_number % N_SPLITS + 1
            seed = MODEL_RANDOM_STATE + target_number * 1_000_000 + model_number * 10_000 + repeat * 100 + fold
            model = seeded_clone(model_templates[model_name], seed)
            model.fit(X_valid[train_local], y_valid[train_local])
            pred = np.asarray(model.predict(X_valid[test_local]), dtype=float).reshape(-1)
            obs = y_valid[test_local]
            repeat_buffers[repeat][test_local] = pred
            repeat_fold_ids[repeat][test_local] = fold

            train_counts = Counter(cohort_valid[train_local])
            test_counts = Counter(cohort_valid[test_local])
            fold_rows.append({
                'Target': target, 'Model': model_name, 'Repeat': repeat, 'Fold': fold,
                'Train_N': len(train_local), 'Validation_N': len(test_local),
                'N_Features': X_valid.shape[1],
                'R2': float(r2_score(obs, pred)),
                'Spearman': safe_spearman(obs, pred),
                'Train_Cohort_Counts': '; '.join(f'{k}:{v}' for k, v in sorted(train_counts.items())),
                'Validation_Cohort_Counts': '; '.join(f'{k}:{v}' for k, v in sorted(test_counts.items())),
            })

        for repeat in range(1, N_REPEATS + 1):
            pred = repeat_buffers[repeat]
            if np.isnan(pred).any() or np.any(repeat_fold_ids[repeat] == 0):
                raise RuntimeError(f'{target}/{model_name}/repeat {repeat} has incomplete OOF predictions.')
            prediction_store[(target, model_name, repeat)] = pred.copy()
            fold_id_store[(target, repeat)] = repeat_fold_ids[repeat].copy()
            repeat_rows.append({
                'Target': target, 'Model': model_name, 'Repeat': repeat,
                'N': len(y_valid), 'N_Features': X_valid.shape[1],
                'R2': float(r2_score(y_valid, pred)),
                'Spearman': safe_spearman(y_valid, pred),
            })
        print(f'  {target:>6s} | {model_name:<20s} completed')
    print(f'Target {target} completed: {len(MODEL_NAMES) * N_REPEATS * N_SPLITS} fits, N={len(y_valid)}')

fold_metrics = pd.DataFrame(fold_rows)
repeat_metrics = pd.DataFrame(repeat_rows)
print(f'Total model fits: {len(fold_metrics):,}')


In [ ]:
fold_summary = (
    fold_metrics.groupby(['Target', 'Model'], as_index=False)
    .agg(
        Fold_R2_Mean=('R2', 'mean'),
        Fold_R2_SD=('R2', 'std'),
        Fold_Spearman_Mean=('Spearman', 'mean'),
        Fold_Spearman_SD=('Spearman', 'std'),
    )
)
repeat_summary = (
    repeat_metrics.groupby(['Target', 'Model'], as_index=False)
    .agg(
        N=('N', 'first'),
        N_Features=('N_Features', 'first'),
        Repeat_OOF_R2_Mean=('R2', 'mean'),
        Repeat_OOF_R2_SD=('R2', 'std'),
        Repeat_OOF_Spearman_Mean=('Spearman', 'mean'),
        Repeat_OOF_Spearman_SD=('Spearman', 'std'),
    )
)
all_model_summary = repeat_summary.merge(fold_summary, on=['Target', 'Model'], validate='one_to_one')
all_model_summary['Target'] = pd.Categorical(all_model_summary['Target'], categories=TARGETS, ordered=True)
all_model_summary = all_model_summary.sort_values(
    ['Target', 'Repeat_OOF_R2_Mean', 'Repeat_OOF_Spearman_Mean'],
    ascending=[True, False, False],
).reset_index(drop=True)
all_model_summary['Rank'] = all_model_summary.groupby('Target', observed=True).cumcount() + 1
all_model_summary['Target'] = all_model_summary['Target'].astype(str)
best_models = all_model_summary[all_model_summary['Rank'] == 1].copy().reset_index(drop=True)

prediction_rows = []
for row in best_models.itertuples(index=False):
    target, model_name = row.Target, row.Model
    valid_global = valid_index_store[target]
    y_valid = y_store[target]
    cohort_valid = cohort_store[target]
    for repeat in range(1, N_REPEATS + 1):
        pred = prediction_store[(target, model_name, repeat)]
        fold_ids = fold_id_store[(target, repeat)]
        prediction_rows.extend({
            'Metaphlan_ID': sample_ids[valid_global[i]], 'Cohort': cohort_valid[i],
            'Target': target, 'Model': model_name, 'Repeat': repeat, 'Fold': int(fold_ids[i]),
            'Observed_Log2': float(y_valid[i]), 'Predicted_Log2': float(pred[i]),
        } for i in range(len(y_valid)))
oof_predictions = pd.DataFrame(prediction_rows)

mean_oof_predictions = (
    oof_predictions.groupby(['Metaphlan_ID', 'Cohort', 'Target', 'Model'], as_index=False)
    .agg(Observed_Log2=('Observed_Log2', 'first'),
         Predicted_Log2_Mean=('Predicted_Log2', 'mean'),
         Predicted_Log2_SD=('Predicted_Log2', 'std'),
         N_Repeats=('Predicted_Log2', 'size'))
)
if not mean_oof_predictions['N_Repeats'].eq(N_REPEATS).all():
    raise RuntimeError(f'Each sample must have {N_REPEATS} OOF predictions from the selected model.')

averaged_rows = []
for (target, model_name), part in mean_oof_predictions.groupby(['Target', 'Model'], sort=False):
    averaged_rows.append({
        'Target': target, 'Model': model_name,
        'Averaged_OOF_R2': float(r2_score(part['Observed_Log2'], part['Predicted_Log2_Mean'])),
        'Averaged_OOF_Spearman': safe_spearman(part['Observed_Log2'].to_numpy(), part['Predicted_Log2_Mean'].to_numpy()),
    })
best_summary = best_models.merge(pd.DataFrame(averaged_rows), on=['Target', 'Model'], validate='one_to_one')
best_summary['Target'] = pd.Categorical(best_summary['Target'], categories=TARGETS, ordered=True)
best_summary = best_summary.sort_values('Target').reset_index(drop=True)
best_summary['Target'] = best_summary['Target'].astype(str)

print('Selected model for each metabolite:')
display(best_summary.round(4))
print('All candidate models ranked within each target:')
display(all_model_summary.round(4))


In [ ]:
fold_metrics.to_csv(OUTPUT_DIR / 'repeated_5x5_fold_metrics.csv', index=False)
repeat_metrics.to_csv(OUTPUT_DIR / 'repeated_5x5_repeat_metrics.csv', index=False)
all_model_summary.to_csv(OUTPUT_DIR / 'repeated_5x5_all_models_summary.csv', index=False)
best_summary.to_csv(OUTPUT_DIR / 'repeated_5x5_best_models_summary.csv', index=False)
oof_predictions.to_csv(OUTPUT_DIR / 'repeated_5x5_oof_predictions.csv', index=False)
mean_oof_predictions.to_csv(OUTPUT_DIR / 'repeated_5x5_mean_oof_predictions.csv', index=False)
print(f'Results saved to: {OUTPUT_DIR.resolve()}')
